# Lab 03 — The vocabulary: PDP, PEP, and what a tier claims

**PART I — Why governed agentic systems are needed**  
*Ch. 4 — Core Concepts and Vocabulary*

`beginner` · about 15 minutes

## By the end of this lab you will be able to

- Separate the policy decision point from the policy enforcement point
- Separate design-time governance from runtime governance
- Distinguish evidence from assurance, and cooperative from independent enforcement
- Position governed agentic systems against neighbouring technology families

**Concepts:** `policy decision point`, `policy enforcement point`, `design-time governance`, `runtime governance`, `evidence vs assurance`, `cooperative enforcement`, `independent enforcement`

---

Run the cell below. It executes the *same* `lab.py` the CLI runs — this notebook is a second view onto one implementation, not a copy, so the two can never disagree.


In [1]:
# Make the repository importable from anywhere under notebooks/
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from nornyx_lab.engine import find_lab, run_lab
meta = find_lab('03')
print(meta.title)

The vocabulary: PDP, PEP, and what a tier claims


## Run the lab


In [2]:
ctx = run_lab(find_lab('03'))

  Lab 03    The vocabulary: PDP, PEP, and what a tier claims

  Textbook: Ch. 4 — Core Concepts and Vocabulary

───────────────────────────────────────────────────────────────────────────────────────────────────────────────────

▸ Decision point vs enforcement point

╭───────────────────────────────────────────── concept · PDP and PEP ─────────────────────────────────────────────╮
│                                                                                                                 │
│  • A policy decision point (PDP) answers may this happen? It computes.                                          │
│  • A policy enforcement point (PEP) sits on the path from intent to effect and applies that answer. It blocks.  │
│                                                                                                                 │
│ The separation is architectural, not cosmetic. A PDP can be perfect and change nothing, because the constraint  │
│ comes from the PEP's position, not from the decision's correctness. Chapter 10: a correct decision does not     │
│ constrain an action whose path to its effect never traverses an element that applies it.                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ nornyx.agentic SPI version : 1.2                                                                                │
│ subject_revision bound     : git:5eed1e55c0ffee1abadcafe0ddba11ed15ea5e11                                       │
│ enforcement model          : cooperative (Tier 2)                                                               │
│                                                                                                                 │
│ It does NOT:                                                                                                    │
│   authenticate the agent      -> identity is asserted by the adapter, not proven                                │
│   authenticate the approver   -> the approval record is supplied, not verified                                  │
│   execute the tool            -> your application does that                                                     │
│   attest that an event is true-> it validates SHAPE and BINDING, not reality                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

▸ Where this sits among things you already run

                                                                                                
 family                        the question it answers         what it does not answer          
 ────────────────────────────────────────────────────────────────────────────────────────────── 
 IAM / identity providers      who is this principal?          may this agent take this action  
                                                               now?                             
 API gateways, egress proxies  may this request leave?         was it authorized under a        
                                                               reviewed policy?                 
 Service meshes                may service A talk to service   may this capability be           
                               B?                              exercised?                       
 Policy engines (OPA, Cedar)   is this request permitted?      who reviewed the policy, and     
                                                               against what evidence?           
 Observability / SIEM          what happened?                  was it allowed to happen?        
 Guardrail services            is this text unsafe?            is this action authorized?       
 Secret managers               may this process read this      may this agent use the tool at   
                               secret?                         all?                             
                                                                                                

None of these is a competitor and none is a substitute. A governed agentic system is the layer  
that names the actor, bounds the action, and binds the evidence — and then hands enforcement to 
whichever of the above sits on the path.

╭────────────────────────────────────────────── ⊘ where this stops ───────────────────────────────────────────────╮
│ The trap this vocabulary exists to prevent. "We use OPA, so our agents are governed." OPA is an excellent PDP.  │
│ Whether your agents are governed depends on where its answers are applied and whether every path to the effect  │
│ goes through that point — which is a question about your architecture, not about OPA.                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── ⚑ your turn ──────────────────────────────────────────────────╮
│ Draw your own system's path from agent intends X to X has happened. Mark every element on that path. Now mark   │
│ which one applies a decision.                                                                                   │
│                                                                                                                 │
│ If the answer is "none", you have a PDP and no PEP — the (a) case above. If the answer is "one, but there are   │
│ three paths", you have Lab 13's problem.                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Inspect what the lab measured

Every lab publishes its findings with `ctx.record(...)`. This is the same data `checks.py` asserts on — poke at it.


In [ ]:
import json
print(json.dumps(ctx.results, indent=2, default=str))

## Prove it

The concept checks for this lab. Each one is a proposition written so a machine can settle it.


In [3]:
import subprocess, sys
checks = ROOT / 'labs' / '03_vocabulary' / 'checks.py'
proc = subprocess.run(
    [sys.executable, '-m', 'pytest', str(checks), '-v', '--no-header'],
    cwd=str(ROOT), capture_output=True, text=True,
    encoding='utf-8', errors='replace',
)
print(proc.stdout[-4000:])

============================= test session starts =============================
collected 6 items

labs\03_vocabulary\checks.py ......                                      [100%]

============================= 6 passed in 29.83s ==============================



## Your turn

The lab printed a **your turn** panel above. Do it here — edit the contract, re-run the cells, and watch which decision changes.

---

Next: [Lab 04 — Identity, capability, and authority](./04_identity_and_capability.ipynb)


In [ ]:
# scratch space
